# Notebook 10 — Score d'alerte boutiques (SB-Z)

**Objectif** : repérer les archétypes **forts en OCG et pas encore sortis en TCG**, et les cartes
déjà disponibles que ces decks joueront.

**Formule** : `alert_score = meta_score_ocg × log(1 + avg_views_week_cartes_core)`

**Filtre « pas encore en TCG » (v3, 2026-11-16)** : un archétype n'est retenu que s'il a au moins
`MIN_OCG_DECKS` decks OCG légaux sur la fenêtre et au moins `MIN_CARTES_INEDITES` cartes clés
jamais sorties en TCG (ni jouées dans un deck TCG, ni présentes dans `cards` avec une `tcg_date`
passée). Jusqu'à la v2 le score mesurait seulement « fort en OCG » : 7 des 10 premiers archétypes
étaient déjà en TCG (Elfnote à 20 % du méta). Le test se fait sur les **cartes**, pas sur le nom de
l'archétype : les noms ne sont pas unifiés (Elfnote existe sous 6 noms en TCG) et un vieil archétype
TCG simplement faible aujourd'hui n'a rien d'un signal.

**Cartes exclues** : staples TCG (> 20 % des decks TCG légaux sur 12 mois et ≥ 30 archétypes),
staples OCG (même règle sur la fenêtre OCG), cartes interdites par la banlist TCG en vigueur
(`banlist_history`, pas `cards.ban_tcg` qui date du dernier `fetch_cards.py`).

**Fenêtre de sortie TCG** : `tcg_window_start` / `tcg_window_end` = date du premier deck OCG des
cartes inédites + 75 à 202 jours (P10-P90 du délai sortie OCG → sortie TCG de 1 897 cartes
2022-2025, médiane 103 jours). Ce n'est pas une date : c'est un intervalle à 80 %, non validé hors
échantillon. L'ancien champ `tcg_entry_estimated` (date du jour + 4 mois, identique pour tous) est supprimé.

**Prérequis** : `python scripts/refresh_decks.py` (decks + `deck_legality` + `banlist_history`).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

DB = os.path.join(os.getcwd(), '..', 'data', 'yugioh.db')
conn = sqlite3.connect(DB)

FENETRE_OCG_MOIS = 3        # fenêtre du méta OCG (une seule borne pour tout le notebook)
MIN_OCG_DECKS = 5           # sous 5 decks, la fréquence d'une carte ne veut rien dire
MIN_CARTES_INEDITES = 3     # absorbe les génériques OCG inédites jouées partout
FREQ_CARTE_CLE = 0.3
FENETRE_STAPLES_MOIS = 12   # 6 mois attrape les pièces de moteur des decks dominants
SEUIL_STAPLE = 0.20
MIN_ARCH_STAPLE = 30        # une staple est jouée par beaucoup d'archétypes, pas par un seul deck dominant
# Délai sortie OCG → sortie TCG, cards.ocg_date → cards.tcg_date, 1 897 cartes 2022-2025 : P10, médiane, P90 (jours)
DELAI_P10, DELAI_MEDIAN, DELAI_P90 = 75, 103, 202

# Date de référence = dernier deck chargé (pas date('now') : le calcul doit être reproductible)
REF = pd.read_sql("SELECT MAX(substr(created, 1, 10)) AS d FROM tournament_decks", conn).d[0]
OCG_DEPUIS = (pd.Timestamp(REF) - pd.DateOffset(months=FENETRE_OCG_MOIS)).strftime('%Y-%m-%d')
TCG_DEPUIS = (pd.Timestamp(REF) - pd.DateOffset(months=FENETRE_STAPLES_MOIS)).strftime('%Y-%m-%d')
LEGAL = "(SELECT deck_id FROM deck_legality WHERE legal = 1)"

print(f'Référence : {REF} | méta OCG depuis {OCG_DEPUIS} | staples TCG depuis {TCG_DEPUIS}')

## Part A — Meta score OCG récent (fenêtre de 3 mois, decks légaux)

In [ ]:
ocg_raw = pd.read_sql(f"""
    SELECT strftime('%Y-%m', created) as month, archetype,
           COUNT(*) as n_decks,
           AVG(CASE WHEN placement IS NOT NULL THEN placement END) as avg_placement
    FROM tournament_decks
    WHERE ocg=1 AND archetype IS NOT NULL AND id IN {LEGAL} AND created >= '{OCG_DEPUIS}'
    GROUP BY month, archetype
""", conn)

agg = ocg_raw.groupby('archetype').agg(
    n_decks=('n_decks', 'sum'),
    avg_placement=('avg_placement', 'mean')
).reset_index()

total = agg['n_decks'].sum()
agg['share_ocg'] = agg['n_decks'] / total
agg['placement_score'] = 1 / agg['avg_placement'].replace(0, np.nan)
agg['placement_score_norm'] = agg['placement_score'] / agg['placement_score'].max()
agg['meta_score_ocg'] = np.sqrt(agg['share_ocg'] * agg['placement_score_norm'].fillna(agg['share_ocg']))
agg = agg.sort_values('meta_score_ocg', ascending=False)

print(f'{total} decks OCG légaux, {len(agg)} archétypes')
print()
print('Top 20 archetypes OCG récents (avant filtre « pas encore en TCG ») :')
print(agg[['archetype', 'n_decks', 'share_ocg', 'meta_score_ocg']].head(20).to_string(index=False))

## Part B — Cartes clés OCG, cartes inédites en TCG, archétypes retenus

In [ ]:
# Cartes du main deck des decks OCG de la fenêtre (même borne que la Part A)
ocg_cards = pd.read_sql(f"""
    SELECT dc.deck_id, dc.card_name, td.archetype
    FROM deck_cards dc JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.ocg=1 AND td.archetype IS NOT NULL AND td.id IN {LEGAL}
      AND td.created >= '{OCG_DEPUIS}' AND dc.zone = 'main'
""", conn)
arch_size = ocg_cards.groupby('archetype')['deck_id'].nunique().rename('n_decks_arch')
n_ocg_decks = ocg_cards['deck_id'].nunique()

# Fréquence par carte par archetype
card_freq = ocg_cards.groupby(['archetype', 'card_name'])['deck_id'].nunique().reset_index(name='n_decks_played')
card_freq = card_freq.join(arch_size, on='archetype')
card_freq['frequency'] = card_freq['n_decks_played'] / card_freq['n_decks_arch']
key_cards = card_freq[card_freq['frequency'] >= FREQ_CARTE_CLE].copy()

# views_week + date de sortie TCG. cards.name porte <P>/<C>/<Q>, deck_cards non : on joint sur le nom nettoyé.
# Une carte absente de `cards` (fetch_cards.py n'interroge que le format TCG) garde views_week = NaN :
# la mettre à 0 pénaliserait précisément les archétypes pas encore sortis.
views = pd.read_sql("""
    SELECT REPLACE(REPLACE(name,'<',''),'>','') AS card_name, views_week, tcg_date FROM cards
""", conn).drop_duplicates('card_name')
key_cards = key_cards.merge(views, on='card_name', how='left')

# Carte « inédite en TCG » : jamais vue dans un deck TCG (toute légalité, Genesys inclus : il prouve
# que la carte existe) et pas de date de sortie TCG passée dans `cards`.
tcg_seen = set(pd.read_sql("""
    SELECT DISTINCT dc.card_name FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id WHERE td.ocg = 0
""", conn)['card_name'])
key_cards['unreleased'] = (~key_cards['card_name'].isin(tcg_seen)
                           & ~(key_cards['tcg_date'].notna() & (key_cards['tcg_date'] <= REF))).astype(int)

# Filtre 1 : staples TCG (decks légaux, 12 mois, > 20 % des decks ET >= 30 archétypes)
staples = pd.read_sql(f"""
    SELECT dc.card_name
    FROM deck_cards dc JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.ocg = 0 AND td.id IN {LEGAL} AND td.created >= '{TCG_DEPUIS}'
    GROUP BY dc.card_name
    HAVING COUNT(DISTINCT dc.deck_id) * 1.0 /
           (SELECT COUNT(*) FROM tournament_decks WHERE ocg = 0 AND id IN {LEGAL} AND created >= '{TCG_DEPUIS}') > {SEUIL_STAPLE}
       AND COUNT(DISTINCT td.archetype) >= {MIN_ARCH_STAPLE}
""", conn)
staple_list = set(staples['card_name'])

# Filtre 1 bis : staples OCG (Maxx "C", The Great Gallant Bandit…). La garde sur le nombre d'archétypes
# est indispensable : sans elle, les pièces du deck dominant (Elfnote) passeraient pour des staples.
occ = ocg_cards.groupby('card_name').agg(n=('deck_id', 'nunique'), a=('archetype', 'nunique'))
staple_ocg_list = set(occ[(occ.n / n_ocg_decks > SEUIL_STAPLE) & (occ.a >= MIN_ARCH_STAPLE)].index)

# Filtre 2 : cartes interdites par la banlist TCG en vigueur
banned_list = set(pd.read_sql("""
    SELECT card_name FROM banlist_history
    WHERE format = 'TCG' AND status = 'Forbidden'
      AND effective_date = (SELECT MAX(effective_date) FROM banlist_history WHERE format = 'TCG')
""", conn)['card_name'])

exclude_list = staple_list | staple_ocg_list | banned_list
key_cards_all = key_cards[~key_cards['card_name'].isin(exclude_list)].copy()

# Archétypes retenus : assez de decks OCG et assez de cartes clés inédites en TCG
per_arch = key_cards_all.groupby('archetype').agg(
    n_ocg_decks=('n_decks_arch', 'first'), n_unreleased=('unreleased', 'sum')).reset_index()
retained = per_arch[(per_arch.n_ocg_decks >= MIN_OCG_DECKS) & (per_arch.n_unreleased >= MIN_CARTES_INEDITES)].copy()

# Fenêtre de sortie TCG : ancre = médiane, sur les cartes inédites de l'archétype, de leur premier deck OCG
# (qui suit la sortie OCG d'un jour en médiane). La médiane évite qu'une vieille exclusivité OCG tire l'ancre.
first_ocg = pd.read_sql("""
    SELECT dc.card_name, MIN(substr(td.created, 1, 10)) AS first_ocg
    FROM deck_cards dc JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.ocg = 1 GROUP BY dc.card_name
""", conn)
unrel = key_cards_all[key_cards_all.unreleased == 1].merge(first_ocg, on='card_name')
unrel['first_ocg'] = pd.to_datetime(unrel['first_ocg'])
anchor = unrel.groupby('archetype')['first_ocg'].median().dt.normalize().rename('anchor')
retained = retained.join(anchor, on='archetype')
retained['ocg_first_seen'] = retained['anchor'].dt.strftime('%Y-%m-%d')
retained['tcg_window_start'] = (retained['anchor'] + pd.Timedelta(days=DELAI_P10)).dt.strftime('%Y-%m-%d')
retained['tcg_window_end'] = (retained['anchor'] + pd.Timedelta(days=DELAI_P90)).dt.strftime('%Y-%m-%d')
retained = retained.drop(columns='anchor')

key_cards_filtered = key_cards_all[key_cards_all['archetype'].isin(retained['archetype'])].copy()

print(f'Staples TCG exclues : {len(staple_list)} | staples OCG : {len(staple_ocg_list)} | interdites TCG : {len(banned_list)}')
print(f'Archétypes OCG : {len(per_arch)} → retenus (pas encore en TCG) : {len(retained)}')
print(f'Cartes clés des archétypes retenus : {len(key_cards_filtered)} '
      f'(dont {int(key_cards_filtered.unreleased.sum())} inédites en TCG)')
print()
print(retained.to_string(index=False))
print()
print('Écartés de justesse (>= 3 cartes inédites mais moins de 5 decks OCG) :')
print(per_arch[(per_arch.n_unreleased >= MIN_CARTES_INEDITES) & (per_arch.n_ocg_decks < MIN_OCG_DECKS)].to_string(index=False))

## Part C — Score d'alerte combiné

In [ ]:
# views_week moyen des cartes clés CONNUES de `cards` (les inédites n'ont pas de vues : NaN ignoré par mean)
views_by_arch = key_cards_filtered.groupby('archetype').agg(
    avg_views_week=('views_week', 'mean'),
    max_views_week=('views_week', 'max'),
    n_key_cards=('card_name', 'count')
).reset_index()

# Fusion avec OCG meta_score, sur les seuls archétypes retenus
alert = agg[['archetype', 'meta_score_ocg', 'share_ocg', 'n_decks']].merge(retained, on='archetype', how='inner')
alert = alert.merge(views_by_arch, on='archetype', how='left')
alert['avg_views_week'] = alert['avg_views_week'].fillna(0)
alert['n_key_cards'] = alert['n_key_cards'].fillna(0)

# Score d'alerte, renormalisé sur les archétypes retenus.
# ⚠ Normalisation relative : le premier vaut toujours 100, même si le signal est faible dans l'absolu.
alert['alert_score'] = alert['meta_score_ocg'] * np.log1p(alert['avg_views_week'])
alert['alert_score_norm'] = (alert['alert_score'] / alert['alert_score'].max() * 100).round(1) if len(alert) else []

alert_sorted = alert.sort_values('alert_score_norm', ascending=False)

# Force OCG de TOUS les archétypes (déjà en TCG ou non), même formule : c'est ce que lit le notebook 11
# (signal_ocg des cartes récentes). Table séparée pour que boutique_alerts ne contienne que le signal boutique.
views_all = key_cards_all.groupby('archetype')['views_week'].mean().rename('avg_views_week').reset_index()
ocg_all = agg[['archetype', 'meta_score_ocg', 'share_ocg', 'n_decks']].merge(views_all, on='archetype', how='left')
ocg_all['avg_views_week'] = ocg_all['avg_views_week'].fillna(0)
ocg_all['ocg_score'] = ocg_all['meta_score_ocg'] * np.log1p(ocg_all['avg_views_week'])
ocg_all['ocg_score'] = (ocg_all['ocg_score'] / ocg_all['ocg_score'].max() * 100).round(1)

print("SCORE D'ALERTE BOUTIQUES — archétypes forts en OCG et pas encore en TCG")
print('=' * 60)
print(alert_sorted[['archetype', 'alert_score_norm', 'meta_score_ocg', 'n_ocg_decks', 'n_unreleased',
                    'avg_views_week', 'tcg_window_start', 'tcg_window_end']].to_string(index=False))

## Part D — Cartes à surveiller par archetype

In [ ]:
print('CARTES CLÉS PAR ARCHETYPE RETENU (staples + interdites exclues)')
print('=' * 65)

for _, a in alert_sorted.iterrows():
    arch = a['archetype']
    print(f"\n🃏 {arch} (alert_score: {a['alert_score_norm']}/100) — sortie TCG probable entre "
          f"{a['tcg_window_start']} et {a['tcg_window_end']} (intervalle à 80 %)")
    cards = key_cards_filtered[key_cards_filtered['archetype'] == arch].sort_values('views_week', ascending=False)
    cards = cards[['card_name', 'frequency', 'views_week', 'unreleased']].head(10).copy()
    cards['frequency'] = (cards['frequency'] * 100).round(0).astype(int).astype(str) + '%'
    print(cards.to_string(index=False))

## Part E — Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

top15 = alert_sorted.head(15)
ax1.barh(top15['archetype'][::-1], top15['alert_score_norm'][::-1], color='#d62728')
ax1.set_xlabel("Score d'alerte (0-100, relatif au premier)")
ax1.set_title(f'Archétypes OCG pas encore en TCG\n(référence : {REF})')

# Tous les archétypes OCG : les retenus en rouge, les autres (déjà en TCG) en gris
ax2.scatter(agg['share_ocg'], agg['meta_score_ocg'], s=20, alpha=0.4, color='grey', label='déjà en TCG / trop peu de decks')
ax2.scatter(alert['share_ocg'], alert['meta_score_ocg'], s=60, color='#d62728', label='retenus')
for _, row in alert.iterrows():
    ax2.annotate(row['archetype'], (row['share_ocg'], row['meta_score_ocg']), fontsize=7, ha='left', va='bottom')
ax2.set_xlabel('Part des decks OCG')
ax2.set_ylabel('Meta score OCG')
ax2.set_title('Force OCG : retenus vs écartés')
ax2.legend(fontsize=8)

plt.tight_layout()
out = os.path.join(os.getcwd(), '..', 'data', 'boutique_alert_score.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print('Graphe sauvegardé : data/boutique_alert_score.png')

## Part F — Sauvegarde en DB

In [ ]:
# Score par archetype (uniquement les archétypes pas encore en TCG)
alert_export = alert_sorted[['archetype', 'alert_score_norm', 'meta_score_ocg', 'share_ocg', 'avg_views_week',
                             'n_key_cards', 'n_ocg_decks', 'n_unreleased',
                             'ocg_first_seen', 'tcg_window_start', 'tcg_window_end']].copy()
alert_export = alert_export.rename(columns={'alert_score_norm': 'alert_score'})
alert_export['computed_at'] = datetime.now().strftime('%Y-%m-%d')
alert_export.to_sql('boutique_alerts', conn, if_exists='replace', index=False)

# Score par carte : toutes les cartes clés des archétypes retenus (plus de top 10 avant filtrage).
# views_week NaN (carte inédite) → card_alert_score 0 : rien à acheter tant qu'elle n'existe pas en TCG.
card_alert = key_cards_filtered.merge(alert[['archetype', 'meta_score_ocg']], on='archetype')
card_alert['views_week'] = card_alert['views_week'].fillna(0)
card_alert['card_alert_score'] = (card_alert['meta_score_ocg'] * np.log1p(card_alert['views_week'])).round(3)
card_alert = card_alert[['archetype', 'card_name', 'frequency', 'views_week', 'card_alert_score', 'unreleased']]
card_alert = card_alert.sort_values('card_alert_score', ascending=False)
card_alert.to_sql('boutique_card_alerts', conn, if_exists='replace', index=False)

# Force OCG de tous les archétypes (lue par le notebook 11)
ocg_export = ocg_all.rename(columns={'n_decks': 'n_ocg_decks'}).sort_values('ocg_score', ascending=False)
ocg_export['computed_at'] = datetime.now().strftime('%Y-%m-%d')
ocg_export.to_sql('ocg_archetype_scores', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print('Tables sauvegardées dans yugioh.db :')
print(f'  boutique_alerts      — {len(alert_export)} archétypes')
print(f'  boutique_card_alerts — {len(card_alert)} cartes (dont {int(card_alert.unreleased.sum())} inédites en TCG)')
print(f'  ocg_archetype_scores — {len(ocg_export)} archétypes OCG (tous, pour le notebook 11)')
print()
print(card_alert.head(10).to_string(index=False))